# IAQ Ontology V2 Builder Notebook

This notebook creates a cleaned V2 ontology for indoor air quality from scratch using rdflib.
It is separate from the graph-ML notebook and only focuses on ontology creation.

In [ ]:
from pathlib import Path
from rdflib import Graph, Namespace, Literal, RDF, RDFS, OWL, XSD, URIRef

BASE_DIR = Path.cwd()
OUT_DIR = BASE_DIR / 'ontology'
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_FILE = OUT_DIR / 'iaq_ontology_v2_from_notebook.ttl'

SOSA = Namespace('http://www.w3.org/ns/sosa/')
SSN = Namespace('http://www.w3.org/ns/ssn/')
TIME = Namespace('http://www.w3.org/2006/time#')
QUDT = Namespace('http://qudt.org/schema/qudt/')
UNIT = Namespace('http://qudt.org/vocab/unit/')
IAQ = Namespace('http://example.org/iaq/')

g = Graph()
g.bind('rdf', RDF)
g.bind('rdfs', RDFS)
g.bind('owl', OWL)
g.bind('xsd', XSD)
g.bind('sosa', SOSA)
g.bind('ssn', SSN)
g.bind('time', TIME)
g.bind('qudt', QUDT)
g.bind('unit', UNIT)
g.bind('iaq', IAQ)

print('Output file:', OUT_FILE)

In [ ]:
def add_class(local_name, label, parent=None, comment=None):
    uri = IAQ[local_name]
    g.add((uri, RDF.type, OWL.Class))
    g.add((uri, RDFS.label, Literal(label, lang='en')))
    if parent is not None:
        g.add((uri, RDFS.subClassOf, parent))
    if comment is not None:
        g.add((uri, RDFS.comment, Literal(comment, lang='en')))
    return uri

def add_object_property(local_name, label, domain=None, range_=None):
    uri = IAQ[local_name]
    g.add((uri, RDF.type, OWL.ObjectProperty))
    g.add((uri, RDFS.label, Literal(label, lang='en')))
    if domain is not None:
        g.add((uri, RDFS.domain, domain))
    if range_ is not None:
        g.add((uri, RDFS.range, range_))
    return uri

def add_data_property(local_name, label, domain=None, range_=None, unit_local=None):
    uri = IAQ[local_name]
    g.add((uri, RDF.type, OWL.DatatypeProperty))
    g.add((uri, RDFS.label, Literal(label, lang='en')))
    if domain is not None:
        g.add((uri, RDFS.domain, domain))
    if range_ is not None:
        g.add((uri, RDFS.range, range_))
    if unit_local is not None:
        g.add((uri, QUDT.hasUnit, UNIT[unit_local]))
    return uri

In [ ]:
ontology_uri = URIRef('http://example.org/iaq/')
g.add((ontology_uri, RDF.type, OWL.Ontology))
g.add((ontology_uri, RDFS.label, Literal('Indoor Air Quality Knowledge Graph Ontology (V2)', lang='en')))
g.add((ontology_uri, OWL.versionInfo, Literal('2.0.0')))
g.add((ontology_uri, OWL.imports, URIRef('http://www.w3.org/ns/sosa/')))
g.add((ontology_uri, OWL.imports, URIRef('http://www.w3.org/ns/ssn/')))
g.add((ontology_uri, OWL.imports, URIRef('http://www.w3.org/2006/time/')))

location = add_class('Location', 'Location', parent=SOSA.Platform)
indoor_location = add_class('IndoorLocation', 'Indoor Location', parent=location)
lab_cls = add_class('Laboratory', 'Laboratory', parent=indoor_location)
apt_cls = add_class('OneRoomApartment', 'One Room Apartment', parent=indoor_location)
sensor_cls = add_class('AirQSensor', 'airQ Sensor', parent=SOSA.Sensor)
obs_cls = add_class('Observation', 'IAQ Observation', parent=SOSA.Observation)
dataset_cls = add_class('Dataset', 'Dataset')
anomaly_cls = add_class('AnomalyEvent', 'Anomaly Event')
add_class('MeasurementError', 'Measurement Error', parent=anomaly_cls)
add_class('PowerOutage', 'Power Outage', parent=anomaly_cls)
add_class('ExternalPollutionEvent', 'External Pollution Event', parent=anomaly_cls)

aq_status_cls = add_class('AirQualityStatus', 'Air Quality Status')
add_class('Good', 'Good', parent=aq_status_cls)
add_class('Moderate', 'Moderate', parent=aq_status_cls)
add_class('Poor', 'Poor', parent=aq_status_cls)
add_class('GasAlarm', 'Gas Alarm', parent=aq_status_cls)
add_class('FireAlarm', 'Fire Alarm', parent=aq_status_cls)

ml_class = add_class('MLPollutionClass', 'ML Pollution Class')
threshold_class = add_class('ThresholdRule', 'Threshold Rule')

add_object_property('measuredAt', 'measured at', domain=obs_cls, range_=location)
add_object_property('hasSourceDataset', 'has source dataset', domain=obs_cls, range_=dataset_cls)
add_object_property('hasAirQualityStatus', 'has air quality status', domain=obs_cls, range_=aq_status_cls)
add_object_property('hasMLClass', 'has ML class', domain=obs_cls, range_=ml_class)
add_object_property('linkedToAnomaly', 'linked to anomaly', domain=obs_cls, range_=anomaly_cls)
add_object_property('usesThreshold', 'uses threshold', domain=ml_class, range_=threshold_class)

add_data_property('buildingName', 'Building Name', range_=XSD.string)
add_data_property('roomType', 'Room Type', range_=XSD.string)
add_data_property('country', 'Country', range_=XSD.string)
add_data_property('city', 'City', range_=XSD.string)
add_data_property('fileName', 'File Name', range_=XSD.string)
add_data_property('serialNumber', 'Serial Number', range_=XSD.string)
add_data_property('installationDate', 'Installation Date', range_=XSD.date)
add_data_property('eventDate', 'Event Date', range_=XSD.date)
add_data_property('eventDescription', 'Event Description', range_=XSD.string)
add_data_property('severity', 'Severity', range_=XSD.string)
add_data_property('notes', 'Notes', range_=XSD.string)
add_data_property('classIndex', 'Class Index', range_=XSD.integer)
add_data_property('thresholdName', 'Threshold Name', range_=XSD.string)
add_data_property('thresholdValue', 'Threshold Value', range_=XSD.double)

In [ ]:
field_schema = [
    ('TypPS', XSD.double, 'Micrometer'),
    ('oxygen', XSD.double, 'PERCENT'),
    ('pm10', XSD.double, 'MicrogramPER_M3'),
    ('cnt0_5', XSD.double, 'UNITLESS'),
    ('co', XSD.double, 'PPM'),
    ('temperature', XSD.double, 'DEG_C'),
    ('performance', XSD.double, 'UNITLESS'),
    ('co2', XSD.double, 'PPM'),
    ('measuretime', XSD.integer, 'MilliSEC'),
    ('so2', XSD.double, 'MicrogramPER_M3'),
    ('no2', XSD.double, 'MicrogramPER_M3'),
    ('cnt5', XSD.double, 'UNITLESS'),
    ('timestamp', XSD.dateTime, None),
    ('pm1', XSD.double, 'MicrogramPER_M3'),
    ('cnt1', XSD.double, 'UNITLESS'),
    ('dewpt', XSD.double, 'DEG_C'),
    ('tvoc', XSD.double, 'PPB'),
    ('pressure', XSD.double, 'HPA'),
    ('cnt10', XSD.double, 'UNITLESS'),
    ('dCO2dt', XSD.double, 'PPM-PER-SEC'),
    ('sound_max', XSD.double, 'DeciBEL'),
    ('health', XSD.double, 'UNITLESS'),
    ('temperature_o2', XSD.double, 'DEG_C'),
    ('cnt2_5', XSD.double, 'UNITLESS'),
    ('o3', XSD.double, 'MicrogramPER_M3'),
    ('humidity', XSD.double, 'PERCENT'),
    ('dHdt', XSD.double, 'GM-PER-M3-PER-SEC'),
    ('humidity_abs', XSD.double, 'GM-PER-M3'),
    ('sound', XSD.double, 'DeciBEL'),
    ('pm2_5', XSD.double, 'MicrogramPER_M3'),
    ('cnt0_3', XSD.double, 'UNITLESS')
]

for field_name, field_type, unit_name in field_schema:
    obs_prop = IAQ[field_name]
    g.add((obs_prop, RDF.type, SOSA.ObservableProperty))
    g.add((obs_prop, RDFS.label, Literal(field_name, lang='en')))
    if unit_name is not None:
        g.add((obs_prop, QUDT.hasUnit, UNIT[unit_name]))

    dp_name = f'{field_name}Value'
    add_data_property(dp_name, dp_name, domain=IAQ.Observation, range_=field_type, unit_local=unit_name)

print('Observable properties added:', len(field_schema))

In [ ]:
# Location individuals
g.add((IAQ.AachenLab, RDF.type, IAQ.Laboratory))
g.add((IAQ.AachenLab, RDFS.label, Literal('Aachen Laboratory', lang='en')))
g.add((IAQ.AachenLab, IAQ.buildingName, Literal('Aachen University')))
g.add((IAQ.AachenLab, IAQ.roomType, Literal('Laboratory')))
g.add((IAQ.AachenLab, IAQ.country, Literal('Germany')))
g.add((IAQ.AachenLab, IAQ.city, Literal('Aachen')))

g.add((IAQ.AachenApartment, RDF.type, IAQ.OneRoomApartment))
g.add((IAQ.AachenApartment, RDFS.label, Literal('Aachen One Room Apartment', lang='en')))
g.add((IAQ.AachenApartment, IAQ.buildingName, Literal('Residential Apartment')))
g.add((IAQ.AachenApartment, IAQ.roomType, Literal('One-Room Apartment')))
g.add((IAQ.AachenApartment, IAQ.country, Literal('Germany')))
g.add((IAQ.AachenApartment, IAQ.city, Literal('Aachen')))

# Dataset individuals
g.add((IAQ.dataset_laboratory_csv, RDF.type, IAQ.Dataset))
g.add((IAQ.dataset_laboratory_csv, IAQ.fileName, Literal('laboratory.csv')))
g.add((IAQ.dataset_one_room_apartement_csv, RDF.type, IAQ.Dataset))
g.add((IAQ.dataset_one_room_apartement_csv, IAQ.fileName, Literal('one_room_apartement.csv')))

# Sensor individuals
g.add((IAQ.airQSensor_Lab, RDF.type, IAQ.AirQSensor))
g.add((IAQ.airQSensor_Lab, SOSA.isHostedBy, IAQ.AachenLab))
g.add((IAQ.airQSensor_Lab, IAQ.serialNumber, Literal('LAB_001')))
g.add((IAQ.airQSensor_Lab, IAQ.installationDate, Literal('2023-01-01', datatype=XSD.date)))

g.add((IAQ.airQSensor_Apartment, RDF.type, IAQ.AirQSensor))
g.add((IAQ.airQSensor_Apartment, SOSA.isHostedBy, IAQ.AachenApartment))
g.add((IAQ.airQSensor_Apartment, IAQ.serialNumber, Literal('APT_001')))
g.add((IAQ.airQSensor_Apartment, IAQ.installationDate, Literal('2023-01-01', datatype=XSD.date)))

# ML classes and thresholds
for local_name, class_idx in [
    ('ml_class_0_clean', 0),
    ('ml_class_1_polluted', 1),
    ('ml_class_2_gas_alarm', 2),
    ('ml_class_3_fire_alarm', 3),
]:
    g.add((IAQ[local_name], RDF.type, IAQ.MLPollutionClass))
    g.add((IAQ[local_name], IAQ.classIndex, Literal(class_idx, datatype=XSD.integer)))

thresholds = [
    ('threshold_CO2_1000ppm', 'CO2 danger threshold', 1000.0),
    ('threshold_CO_200ppm', 'CO danger threshold', 200.0),
    ('threshold_PM25_35ugm3', 'PM2.5 danger threshold', 35.0),
    ('threshold_TVOC_500ppb', 'TVOC danger threshold', 500.0),
]
for local_name, name, value in thresholds:
    g.add((IAQ[local_name], RDF.type, IAQ.ThresholdRule))
    g.add((IAQ[local_name], IAQ.thresholdName, Literal(name)))
    g.add((IAQ[local_name], IAQ.thresholdValue, Literal(value, datatype=XSD.double)))

# Corrected anomaly events
anomalies = [
    ('event_20230417_PowerOutage', IAQ.PowerOutage, '2023-04-17', 'Lab power outage, likely short circuit', 'High'),
    ('event_20230521_ExternalFireEvent', IAQ.ExternalPollutionEvent, '2023-05-21', 'Large fire 9-10 km away from measurement location', 'High'),
    ('event_20230709_HumiditySensorError', IAQ.MeasurementError, '2023-07-09', 'Humidity spike caused fine dust measurement artifacts', 'Medium')
]
for local_name, event_class, event_date, desc, severity in anomalies:
    e = IAQ[local_name]
    g.add((e, RDF.type, event_class))
    g.add((e, IAQ.eventDate, Literal(event_date, datatype=XSD.date)))
    g.add((e, IAQ.eventDescription, Literal(desc)))
    g.add((e, IAQ.severity, Literal(severity)))
    g.add((e, IAQ.affectedSensor, IAQ.airQSensor_Lab))
    g.add((e, IAQ.affectedLocation, IAQ.AachenLab))

print('Core individuals and anomaly events added.')

In [ ]:
g.serialize(destination=str(OUT_FILE), format='turtle')
print('Ontology V2 generated at:', OUT_FILE)
print('Total triples:', len(g))

In [ ]:
preview_lines = OUT_FILE.read_text(encoding='utf-8').splitlines()[:40]
for line in preview_lines:
    print(line)